In [ ]:
#| default_exp harness

# harness

> Getting the record out of whichever agent is running.

Every harness reports what it is doing in its own shape. An adapter turns one of those shapes
into a list of calls on a `Scribe`, and `ingest` makes them. Adding a harness is one function.

Nothing here decides where the ledger is. The adapter reports the working directory the harness
gave it, and `find_home` walks up from there, so three harnesses in one repository land on one
file without any of them being configured.

In [ ]:
#| export
import json, os, re, sys
from pathlib import Path

from fastcore.basics import AttrDict
from fastcore.foundation import L

from panjika.core import Home, now
from panjika.write import Scribe, action_for

## What an adapter returns

A `Plan` is the session it belongs to, where it happened, and the calls to make. Nothing is
written while an adapter runs, so an adapter is a pure function and a test for one needs no
disk.

In [ ]:
#| export
def plan(session='', start='', **acts):
    "Empty plan for one session."
    return AttrDict(session=str(session or ''), start=str(start or ''), acts=L(), **acts)


def act(p, do, **kw):
    "Add one call to a plan. `do` names a `Scribe` method."
    p.acts.append(AttrDict(do=do, **kw))
    return p


def first_of(d, *keys, default=''):
    "The first key present and non-empty in `d`."
    for k in keys:
        v = (d or {}).get(k)
        if v not in (None, '', [], {}): return v
    return default

## Claude Code

Written against the documented hook payloads. `SessionStart` opens the session, `PostToolUse`
and `PostToolUseFailure` record the calls, `SessionEnd` closes it. A tool call that names a file
and means to write to it also records a touch, which is what the landing check reads back.

In [ ]:
#| export
#: Where a file path hides in a tool's arguments, across the harnesses and their MCP servers.
PATH_KEYS = ('file_path', 'filePath', 'notebook_path', 'notebookPath', 'path', 'filename', 'file')

#: What else is worth putting in the one-line target when there is no path.
TARGET_KEYS = ('command', 'url', 'pattern', 'query', 'prompt', 'description', 'code')


def tool_target(args):
    "The one thing a tool call was pointed at: a path, else a command, a url, a pattern."
    return str(first_of(args, *PATH_KEYS, *TARGET_KEYS))


def tool_path(args):
    "The file a tool call names, or `''`."
    return str(first_of(args, *PATH_KEYS))

In [ ]:
#| export
def claude_code(payload):
    "One Claude Code hook payload as a plan. See the hooks reference for the field names."
    p = payload or {}
    event = str(p.get('hook_event_name') or '')
    out = plan(p.get('session_id'), p.get('cwd'))
    args = p.get('tool_input') or {}
    tool = str(p.get('tool_name') or '')
    if event == 'SessionStart':
        act(out, 'begin', harness='claude-code', model=str(p.get('model') or ''),
            agent=str(p.get('agent_type') or ''), parent=str(p.get('agent_id') and p.get('session_id') or ''))
    elif event == 'UserPromptSubmit':
        text = str(first_of(p, 'user_input', 'prompt'))
        act(out, 'note', text=text)
        act(out, 'write', kind='session', prompt=text[:2000])
    elif event in ('PostToolUse', 'PostToolUseFailure'):
        ok = event == 'PostToolUse'
        act(out, 'step', tool=tool, target=tool_target(args), ok=ok, args=args,
            output=p.get('tool_response'), summary='' if ok else 'the tool failed')
        path = tool_path(args)
        if ok and path and action_for(tool) == 'write':
            act(out, 'touch', path=path, action='edit')
    elif event in ('Stop', 'SubagentStop'):
        last = str(p.get('last_assistant_message') or '')
        if last: act(out, 'note', text=last[:4000])
    elif event == 'SessionEnd':
        act(out, 'end', status='done', reason=str(p.get('reason') or ''))
    return out

## Codex

The payload shape here is **not verified**. Codex's `notify` program and its lifecycle hooks
were not documented anywhere reachable when this was written, so the adapter reads several
spellings of each field and records what it finds. It is one function; correct it against your
own Codex version and nothing else moves.

In [ ]:
#| export
#: Codex's own tool names. `apply_patch` is its editor, and it names the files inside the
#: patch envelope rather than in an argument.
CODEX_EDITS = ('apply_patch', 'Edit', 'Write')
_PATCH_FILE = re.compile(r'^\*\*\* (?:Add|Update|Delete) File: (.+)$', re.M)


def patch_paths(text):
    "The files an `apply_patch` envelope names."
    return [m.group(1).strip() for m in _PATCH_FILE.finditer(str(text or ''))]


def codex_paths(tool, args):
    "The files one Codex tool call moved."
    if tool in CODEX_EDITS:
        body = first_of(args, 'command', 'input', 'patch', 'content')
        if (found := patch_paths(body)): return found
    path = tool_path(args)
    return [path] if path and action_for(tool) == 'write' else []


def _codex_failed(response):
    "Whether a `tool_response` reports an error. Codex has no separate failure event."
    if isinstance(response, dict):
        return bool(response.get('isError') or response.get('is_error') or response.get('error'))
    return False


def codex(payload):
    """One Codex lifecycle-hook payload as a plan.

    Codex hooks deliver one JSON object on stdin and use the same event names as Claude Code,
    so this reads much the same. What differs: there is no separate failure event, so an error
    is read out of `tool_response`; the editor is `apply_patch`, which names its files inside
    the patch rather than in an argument; and a subagent reports its parent's `session_id`
    alongside its own `agent_id`.

    Verified against the hook reference at https://developers.openai.com/codex/hooks.
    """
    p = payload or {}
    event = str(p.get('hook_event_name') or '')
    out = plan(p.get('session_id'), p.get('cwd'))
    args = p.get('tool_input') or {}
    tool = str(p.get('tool_name') or '')
    agent = str(p.get('agent_type') or p.get('agent_id') or '')
    if event in ('SessionStart', 'SubagentStart'):
        act(out, 'begin', harness='codex', model=str(p.get('model') or ''), agent=agent,
            parent=str(p.get('session_id') or '') if p.get('agent_id') else '')
    elif event == 'UserPromptSubmit':
        text = str(first_of(p, 'prompt', 'user_input', 'user_prompt'))
        act(out, 'note', text=text)
        act(out, 'write', kind='session', prompt=text[:2000], origin='human')
    elif event == 'PostToolUse':
        resp = p.get('tool_response')
        ok = not _codex_failed(resp)
        act(out, 'step', tool=tool, target=tool_target(args), ok=ok, args=args, output=resp,
            summary='' if ok else 'the tool reported an error')
        if ok:
            for path in codex_paths(tool, args): act(out, 'touch', path=path, action='edit')
    elif event in ('Stop', 'SubagentStop'):
        text = str(first_of(p, 'last_assistant_message', 'last-assistant-message'))
        if text: act(out, 'note', text=text[:4000])
    elif event == 'SessionEnd':
        act(out, 'end', status='done', reason=str(p.get('reason') or ''))
    return out


def codex_notify(payload):
    """One legacy Codex `notify` payload as a plan.

    `notify` fires once per completed turn and nowhere else, so it records a turn and never a
    tool call. Its keys are kebab-case, unlike every other Codex surface, and it carries no
    model. Codex passes it as a command-line argument with stdin closed, so it is reached
    through `panjika record`, not `panjika hook`.

    Verified against `codex-rs/hooks/src/legacy_notify.rs`, which serialises
    `AgentTurnComplete` with `#[serde(rename_all = "kebab-case")]`.
    """
    p = payload or {}
    if str(p.get('type') or '') != 'agent-turn-complete': return plan('', '')
    # `thread-id` and `cwd` were added after this payload first shipped, so an older Codex
    # sends neither and the turn id is all there is to join on.
    out = plan(first_of(p, 'thread-id', 'turn-id'), str(p.get('cwd') or ''))
    act(out, 'begin', harness='codex', model='', title=str(p.get('client') or ''))
    for text in (p.get('input-messages') or []):
        act(out, 'write', kind='session', prompt=str(text)[:2000], origin='human')
    if (last := p.get('last-assistant-message')): act(out, 'note', text=str(last)[:4000])
    act(out, 'end', status='turn-complete', turn=str(p.get('turn-id') or ''))
    return out

## Ramabana

Ramabana already assembles the whole of a turn in one record before it writes it to its own
log: the prompt, the reply, the model, the usage, and every tool call with its arguments and its
result. That record is the natural thing to hand over, so this adapter takes it whole and one
turn becomes a session's worth of ledger.

On Ramabana's side that is three lines at the end of `Agent._remember`.

In [ ]:
#| export
def ramabana(payload):
    """One Ramabana turn record as a plan.

    `{session, model, prompt, reply, at, usage, activity: [Act.dict(), ...]}`, which is what
    `Agent._remember` builds. An `Act` on its own is taken as one step.
    """
    p = payload or {}
    out = plan(first_of(p, 'session', 'session_id'), first_of(p, 'cwd', 'root'))
    if not any(p.get(k) for k in ('activity', 'prompt', 'reply', 'tool', 'model', 'error')):
        return out
    if p.get('tool') and 'activity' not in p:          # one `Act`, not a whole turn
        act(out, 'step', tool=p['tool'], target=_act_target(p), ok=bool(p.get('ok', True)),
            secs=float(p.get('secs') or 0), summary=str(p.get('summary') or ''),
            args=p.get('args'), output=p.get('detail'))
        if _act_path(p): act(out, 'touch', path=_act_path(p), action='edit')
        return out
    usage, at = p.get('usage') or {}, p.get('at')
    # `at` is when the turn ran. Ramabana hands a turn over as it finishes, but it also replays
    # its own history, and a session stamped at ingest sorts as the newest thing that ever
    # happened -- which is what `log`, `--since` and a bare `landed` all resolve through.
    act(out, 'write', kind='session', harness='ramabana', model=str(p.get('model') or ''),
        prompt=str(p.get('prompt') or '')[:2000], status=str(p.get('state') or 'done'),
        at=at, started=at,
        tokens_in=usage.get('input'), tokens_out=usage.get('output'), cost=usage.get('cost'))
    for a in (p.get('activity') or ()):
        act(out, 'step', tool=str(a.get('tool') or ''), target=_act_target(a),
            ok=bool(a.get('ok', True)), secs=float(a.get('secs') or 0),
            summary=str(a.get('summary') or ''), args=a.get('args'), output=a.get('detail'))
        if a.get('ok', True) and _act_path(a): act(out, 'touch', path=_act_path(a), action='edit')
    if p.get('reply'): act(out, 'note', text=str(p['reply'])[:4000])
    if p.get('error'): act(out, 'note', text=f"failed: {p['error']}")
    return out


def _act_target(a): return tool_target(a.get('args') or {}) or str(a.get('summary') or '')


def _act_path(a):
    "The file a Ramabana act names, when the tool it ran was one that writes."
    path = tool_path(a.get('args') or {})
    return path if path and action_for(a.get('tool') or '') == 'write' else ''


## Anything else

The generic adapter is the contract for a harness with no adapter of its own: name the session,
name the call, and it is recorded. `panjika record` on the command line goes through this, so a
shell script is a first-class harness.

In [ ]:
#| export
def generic(payload):
    """A payload that already speaks the ledger's own shape.

    `{"session": "...", "cwd": "...", "do": "step", "tool": "Edit", "target": "a.py"}`, or the
    same with `acts` holding a list of those.
    """
    p = dict(payload or {})
    out = plan(first_of(p, 'session', 'session_id'), first_of(p, 'cwd', 'start'))
    acts = p.get('acts')
    if acts is None:
        do = p.pop('do', None)
        for k in ('session', 'session_id', 'cwd', 'start', 'harness_name'): p.pop(k, None)
        if not do and not p: return out         # nothing was asked for, so nothing is written
        acts = [{'do': do or 'note', **p}]
    for a in acts:
        a = dict(a)
        act(out, a.pop('do', 'note'), **a)
    return out


#: Every harness this package can read. A new one is a function and a line here.
ADAPTERS = {'claude-code': claude_code, 'codex': codex, 'codex-notify': codex_notify, 'ramabana': ramabana,
            'generic': generic}

## Making the calls

In [ ]:
#| export
def ingest(payload, adapter='generic', home=None, start='.', harness=''):
    "Apply one payload to a ledger. Returns the plan that was applied."
    build = ADAPTERS.get(str(adapter), generic)
    p = build(payload)
    # a payload an adapter had nothing to say about must not make a ledger where there is none
    if not p.acts:
        p.wrote, p.home = 0, ''
        return p
    sc = Scribe(home=home, session=p.session, start=p.start or start)
    if not sc.home.exists: sc.home.init()
    for a in p.acts:
        do = a.pop('do')
        if do == 'begin': a.setdefault('harness', harness or str(adapter))
        getattr(sc, do)(**a)
    p.wrote, p.home, p.session = len(p.acts), str(sc.home.path), sc.session
    return p


def hook(adapter='generic', stream=None, home=None, start=None):
    """Read one payload from `stream` and record it. Never raises.

    A hook that fails must not take the harness down with it, and a harness that cannot be
    recorded is still a harness that works, so every failure here is swallowed and reported on
    stderr where the harness logs it.
    """
    try:
        raw = (stream or sys.stdin).read()
        payload = json.loads(raw) if raw.strip() else {}
    except Exception as e:
        print(f'panjika: unreadable payload ({e})', file=sys.stderr)
        return None
    if not payload: return None
    try: return ingest(payload, adapter, home, start or payload.get('cwd') or os.getcwd())
    except Exception as e:
        print(f'panjika: {type(e).__name__}: {e}', file=sys.stderr)
        return None

## Trying it

In [ ]:
import subprocess, tempfile
from fastcore.test import test_eq
from panjika.read import Ledger

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'proj'; d.mkdir(parents=True)
_git(d, 'init', '-q', '-b', 'main')
_git(d, 'config', 'user.email', 'a@b.c'); _git(d, 'config', 'user.name', 'T')
(d/'app.py').write_text('X = 1\n')
_git(d, 'add', '-A'); _git(d, 'commit', '-qm', 'first')
Home(d/'.panjika').init()

def cc(**kw): return ingest({'session_id': 'cc-1', 'cwd': str(d), **kw}, 'claude-code', d/'.panjika')

cc(hook_event_name='SessionStart', model='opus-5')
cc(hook_event_name='UserPromptSubmit', user_input='bump the constant')
(d/'app.py').write_text('X = 2\n')
cc(hook_event_name='PostToolUse', tool_name='Edit',
   tool_input={'file_path': str(d/'app.py')}, tool_response='ok')
cc(hook_event_name='PostToolUseFailure', tool_name='Bash',
   tool_input={'command': 'pytest'}, tool_response='2 failed')
cc(hook_event_name='SessionEnd', reason='clear')

row = Ledger(d/'.panjika').session('cc-1')
test_eq((row.harness, row.model, row.status), ('claude-code', 'opus-5', 'done'))
test_eq(row.prompt, 'bump the constant')
test_eq((row.steps_ok, row.steps_fail), (1, 1))
test_eq([t.path for t in row.files], ['app.py'])

In [ ]:
# one whole Ramabana turn, handed over as the record Ramabana already builds
ingest({'session': 'rb-1', 'cwd': str(d), 'model': 'sonnet', 'prompt': 'tidy the imports',
        'reply': 'done, two files', 'usage': {'input': 1840, 'output': 96},
        'activity': [{'tool': 'search_code', 'args': {'query': 'import'}, 'ok': True, 'secs': 0.2},
                     {'tool': 'edit_file', 'args': {'path': str(d/'app.py')}, 'ok': True, 'secs': 0.4},
                     {'tool': 'run_shell', 'args': {'command': 'pytest -q'}, 'ok': False}]},
       'ramabana', d/'.panjika')
row = Ledger(d/'.panjika').session('rb-1')
test_eq((row.harness, row.model), ('ramabana', 'sonnet'))
test_eq((row.n_steps, row.steps_fail), (3, 1))
test_eq([t.path for t in row.files], ['app.py'])   # only the tool that writes leaves a touch

In [ ]:
# the same ledger, from a harness that has no adapter at all
ingest({'session': 'sh-1', 'cwd': str(d), 'do': 'begin', 'harness': 'a shell script',
        'prompt': 'regenerate the fixtures'}, 'generic', d/'.panjika')
ingest({'session': 'sh-1', 'cwd': str(d), 'do': 'step', 'tool': 'make', 'target': 'fixtures'},
       'generic', d/'.panjika')
test_eq([s.harness for s in Ledger(d/'.panjika').sessions()],
        ['a shell script', 'ramabana', 'claude-code'])

In [ ]:
# Ramabana describes one session with a record per turn and never calls `begin`, so nothing
# writes `started` for it and nothing but the turn itself knows when it ran. A session stamped
# at ingest sorts as the newest thing that ever happened, and a reader that subtracts a missing
# `started` from an epoch reports the age of the epoch as the length of the session.
JAN = 1768460400.0                                    # 2026-01-15, long before this cell runs
def _turn(at, prompt, **kw):
    return {'session': 'rb-when', 'cwd': str(d), 'at': at, 'model': 'sonnet', 'prompt': prompt,
            'usage': {'input': 10, 'output': 2}, **kw}

ingest(_turn(JAN, 'round the vat'), 'ramabana', d/'.panjika')
ingest(_turn(JAN + 300, 'and add a test'), 'ramabana', d/'.panjika')

row = Ledger(d/'.panjika').session('rb-when')
test_eq(row.started, JAN)                             # the first turn, not the last and not now
test_eq(row.prompt, 'round the vat')
test_eq(row.seconds, 300.0)                           # not the age of the epoch

# and it sorts by when it ran, so it does not displace a session that really is the newest
assert Ledger(d/'.panjika').sessions()[0].session != 'rb-when' 

### What a payload looks like

An adapter turns one harness payload into a plan and writes nothing, so the shape each
harness sends can be shown here rather than described.

In [ ]:
# Claude Code: one JSON object on stdin, per event.
p = claude_code({'hook_event_name': 'PostToolUse', 'session_id': 'cc-1', 'cwd': '/repo',
                 'tool_name': 'Edit', 'tool_input': {'file_path': 'charges.py'},
                 'tool_response': 'ok'})
test_eq([a.do for a in p.acts], ['step', 'touch'])
test_eq(p.acts[1].path, 'charges.py')

# Codex: the same events and the same delivery, but the editor is `apply_patch`, which names
# the files it touches inside the patch envelope rather than in an argument.
envelope = '*** Begin Patch\n*** Update File: charges.py\n@@\n-a\n+b\n*** End Patch\n'
p = codex({'hook_event_name': 'PostToolUse', 'session_id': 'cdx-1', 'cwd': '/repo',
           'model': 'gpt-5.6', 'tool_name': 'apply_patch', 'tool_use_id': 'c1',
           'tool_input': {'command': envelope}, 'tool_response': 'done'})
test_eq([a.path for a in p.acts if a.do == 'touch'], ['charges.py'])

# Codex has no separate failure event, so an error is read out of the response.
p = codex({'hook_event_name': 'PostToolUse', 'session_id': 'cdx-1', 'tool_name': 'Bash',
           'tool_input': {'command': 'pytest'}, 'tool_response': {'isError': True}})
assert not p.acts[0].ok
test_eq([a for a in p.acts if a.do == 'touch'], [])

# The legacy `notify` route is kebab-case, arrives as an argument rather than on stdin, and
# fires once per turn, so it records a turn and never a change.
p = codex_notify({'type': 'agent-turn-complete', 'thread-id': 'th-1', 'turn-id': '99',
                  'cwd': '/repo', 'input-messages': ['rename foo to bar'],
                  'last-assistant-message': 'renamed'})
test_eq(p.session, 'th-1')
test_eq([a for a in p.acts if a.do in ('step', 'touch')], [])

In [ ]:
# a hook is given a broken payload and the harness carries on regardless
import io
test_eq(hook('claude-code', io.StringIO('{not json')), None)
test_eq(hook('claude-code', io.StringIO('')), None)
test_eq(hook('claude-code', io.StringIO('{}')), None)

# an event no adapter reads writes nothing at all, and makes no ledger where there was none
test_eq(ingest({'session_id': 'x', 'hook_event_name': 'PreCompact'}, 'claude-code').wrote, 0)

## Installing the hooks

`install` writes the configuration each harness reads. Claude Code takes a `settings.json` in
the project, git takes a `post-commit` script, and Codex is printed rather than written because
its shape could not be verified.

The Claude Code entries are merged into whatever is already there. An entry `panjika` already
wrote is replaced; anything else is left alone.

In [ ]:
#| export
CC_EVENTS = {'SessionStart': '', 'UserPromptSubmit': '', 'SessionEnd': '',
             'PostToolUse': '*', 'PostToolUseFailure': '*'}

POST_COMMIT = """#!/bin/sh
# Written by `panjika install --git`. Links each commit to the agent sessions that earned it.
panjika link-commit "$(git rev-parse HEAD)" >/dev/null 2>&1 || true
"""

#: Codex's lifecycle events, and what each one matches. Its vocabulary is Claude Code's, with
#: no separate failure event: a failed call is a `PostToolUse` whose `tool_response` says so.
CODEX_EVENTS = {'SessionStart': '', 'UserPromptSubmit': '', 'SessionEnd': '',
                'PostToolUse': '.*', 'SubagentStart': '', 'SubagentStop': ''}

CODEX_TRUST = ("Codex will not run a hook until you have reviewed it. Run `/hooks` in Codex, "
               "read the entry, and trust it.")

#: The legacy `notify` route, for a Codex too old to have hooks. It fires once per turn and
#: never per tool call, and Codex passes it as an argument with stdin closed, so it goes
#: through `panjika record` rather than `panjika hook`.
CODEX_SNIPPET = """# ~/.codex/config.toml
# Only for a Codex without lifecycle hooks. One record per turn, no tool calls.
notify = ["panjika", "record", "--adapter", "codex-notify"]
"""


def codex_hooks(path='.codex/hooks.json', command='panjika hook codex'):
    "The Codex hooks document with panjika's hooks merged into it."
    p = Path(path)
    try: doc = json.loads(p.read_text()) if p.exists() else {}
    except Exception: doc = {}
    if not isinstance(doc, dict): doc = {}
    hooks = doc.setdefault('hooks', {})
    for event, matcher in CODEX_EVENTS.items():
        groups = [g for g in hooks.get(event, [])
                  if not any('panjika' in str(h.get('command', ''))
                             for h in (g.get('hooks') or []))]
        group = {'hooks': [{'type': 'command', 'command': command, 'timeout': 10}]}
        if matcher: group['matcher'] = matcher
        hooks[event] = groups + [group]
    return doc


def cc_settings(path='.claude/settings.json', command='panjika hook claude-code'):
    "The Claude Code settings document with panjika's hooks merged into it."
    p = Path(path)
    try: doc = json.loads(p.read_text()) if p.exists() else {}
    except Exception: doc = {}
    if not isinstance(doc, dict): doc = {}
    hooks = doc.setdefault('hooks', {})
    for event, matcher in CC_EVENTS.items():
        entries = [e for e in hooks.get(event, [])
                   if not any('panjika' in str(h.get('command', ''))
                              for h in (e.get('hooks') or []))]
        entry = {'hooks': [{'type': 'command', 'command': command, 'timeout': 10}]}
        if matcher: entry['matcher'] = matcher
        hooks[event] = entries + [entry]
    return doc

In [ ]:
#| export
def install(root='.', claude_code=True, git=True, codex=True, home=None):
    "Write the hook configuration into `root`. Returns what was written and what was not."
    root = Path(root)
    out, ledger = AttrDict(wrote=L(), skipped=L(), codex=CODEX_SNIPPET,
                           note=CODEX_TRUST), Home(home, root)
    ledger.init()
    out.home = str(ledger.path)
    if claude_code:
        p = root/'.claude'/'settings.json'
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps(cc_settings(p), indent=2) + '\n')
        out.wrote.append(str(p))
    if codex:
        p = root/'.codex'/'hooks.json'
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps(codex_hooks(p), indent=2) + '\n')
        out.wrote.append(str(p))
    if git:
        from panjika.core import git_root
        gr = git_root(root)
        if gr is None: out.skipped.append('post-commit: not inside a git repository')
        else:
            p = Path(gr)/'.git'/'hooks'/'post-commit'
            if p.exists() and 'panjika' not in p.read_text():
                out.skipped.append(f'post-commit: {p} exists and is not ours')
            else:
                p.parent.mkdir(parents=True, exist_ok=True)
                p.write_text(POST_COMMIT)
                p.chmod(0o755)
                out.wrote.append(str(p))
    return out

In [ ]:
# installing twice leaves one panjika hook per event, not two
out = install(d, home=d/'.panjika')
out = install(d, home=d/'.panjika')
doc = json.loads((d/'.claude'/'settings.json').read_text())
for event in CC_EVENTS:
    entries = [h for e in doc['hooks'][event] for h in e['hooks'] if 'panjika' in h['command']]
    test_eq(len(entries), 1)
assert (Path(d)/'.git'/'hooks'/'post-commit').exists()

In [ ]:
# somebody else's hook on the same event is left where it is
doc = json.loads((d/'.claude'/'settings.json').read_text())
doc['hooks']['PostToolUse'].append({'matcher': 'Write', 'hooks': [{'type': 'command', 'command': 'ruff format'}]})
(d/'.claude'/'settings.json').write_text(json.dumps(doc))
install(d, home=d/'.panjika')
doc = json.loads((d/'.claude'/'settings.json').read_text())
commands = [h['command'] for e in doc['hooks']['PostToolUse'] for h in e['hooks']]
test_eq(sorted(commands), ['panjika hook claude-code', 'ruff format'])

In [ ]:
# Codex gets a hooks document of its own, merged the same way, and Codex will not run any of
# it until you have reviewed it with `/hooks`.
doc = json.loads((d/'.codex'/'hooks.json').read_text())
for event in CODEX_EVENTS:
    entries = [h for g in doc['hooks'][event] for h in g['hooks'] if 'panjika' in h['command']]
    test_eq(len(entries), 1)
test_eq(doc['hooks']['PostToolUse'][0]['matcher'], '.*')
assert 'trust' in out.note

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()